# 실험 6. MLP (PyTorch)

- 목적: 트리 모델과 방식이 다른 신경망의 OOF 예측을 만들어 LightGBM과 앙상블 효과 확인
- 실행: Kaggle GPU 노트북 (로컬에는 `torch` 미설치)
- 산출물: `oof_mlp.csv`, `test_mlp.csv` (id, fold, pred)

## 환경설정

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
START_TIME = time.time()
np.random.seed(SEED)
torch.manual_seed(SEED)

# A GPU helps big networks; this small MLP trains fine on CPU (Kaggle: Settings -> Accelerator for a GPU).
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

# Chart style: thin marks, light grid, no top/right frame.
COLOR_MAIN = "#2a78d6"   # blue   - single-series charts, "satisfied"
COLOR_ALT = "#eb6834"    # orange - "not satisfied" / second series
COLOR_MUTED = "#8a8986"  # gray   - reference lines
plt.rcParams.update({
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3df",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "axes.titleweight": "bold",
})

## 데이터 경로 탐색 및 로드

In [ ]:
REQUIRED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]


def has_all_files(folder: Path) -> bool:
    return all((folder / name).is_file() for name in REQUIRED_FILES)


def find_data_dir() -> Path:
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates = sorted({p.parent for p in kaggle_root.glob("**/train.csv") if has_all_files(p.parent)})
    else:
        local = [p.resolve() for p in (Path("data"), Path("../data")) if has_all_files(p)]
        candidates = list(dict.fromkeys(local))  # drop duplicates, keep order

    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            f"데이터 폴더를 찾지 못했습니다. {REQUIRED_FILES} 세 파일이 모두 있는 폴더가 필요합니다.\n"
            "- Kaggle: 오른쪽 Input 패널에서 대회 데이터를 추가하세요.\n"
            "- 로컬: 노트북과 같은 위치의 data/ 폴더에 파일을 넣으세요."
        )
    raise RuntimeError(
        "데이터 폴더 후보가 여러 개입니다. 사용할 폴더를 DATA_DIR에 직접 지정하세요:\n"
        + "\n".join(f"  - {c}" for c in candidates)
    )


DATA_DIR = find_data_dir()
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

TARGET = "satisfaction"
CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]
NUM_COLS = [c for c in train.columns if c not in CAT_COLS + ["id", TARGET]]
RATING_COLS = [c for c in NUM_COLS if train[c].max() <= 5]
y = train[TARGET].astype(int).values

print("DATA_DIR =", DATA_DIR)
print("train", train.shape, "| test", test.shape)

## 교차검증 Fold 및 예측 저장 함수

In [ ]:
# Where this notebook writes its predictions (Kaggle only allows writing under /kaggle/working).
OUT_DIR = Path("/kaggle/working/outputs") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR =", OUT_DIR.resolve())


N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(train, y))

# fold_id[i] = the fold in which row i is used for validation (same in every notebook of the series).
fold_id = np.full(len(train), -1)
for k, (_, va_idx) in enumerate(FOLDS):
    fold_id[va_idx] = k
assert (fold_id >= 0).all()


def save_predictions(name: str, oof: np.ndarray, test_pred: np.ndarray) -> None:
    """Save OOF and test predictions keyed by id, so notebook 07 can align them safely."""
    assert np.isfinite(oof).all() and np.isfinite(test_pred).all()
    pd.DataFrame({"id": train["id"].values, "fold": fold_id, "pred": oof}).to_csv(OUT_DIR / f"oof_{name}.csv", index=False)
    pd.DataFrame({"id": test["id"].values, "pred": test_pred}).to_csv(OUT_DIR / f"test_{name}.csv", index=False)
    print(f"saved oof_{name}.csv / test_{name}.csv")


FREQ_COLS = ["Age", "Flight Distance"]


def add_freq_features(fit_df: pd.DataFrame, *apply_dfs: pd.DataFrame) -> list:
    """Frequency encoding fit on fit_df only (no peeking at validation/test), applied to every frame."""
    out = []
    for df in (fit_df, *apply_dfs):
        df = df.copy()
        for col in FREQ_COLS:
            counts = fit_df[col].value_counts()
            df[f"{col}_freq"] = df[col].map(counts).fillna(0).astype(float)
        out.append(df)
    return out

## 데이터 전처리 파이프라인

In [ ]:
X = train.drop(columns=["id", TARGET])
X_test = test.drop(columns=["id"])
for col in CAT_COLS:
    categories = sorted(train[col].unique())
    X[col] = pd.Categorical(X[col], categories=categories)
    X_test[col] = pd.Categorical(X_test[col], categories=categories)

In [ ]:
def prepare_fold(tr_idx, va_idx):
    X_tr, X_va, X_te = add_freq_features(X.iloc[tr_idx], X.iloc[va_idx], X_test)
    num_cols = [c for c in X_tr.columns if c not in CAT_COLS]
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
    ])
    arrays = [prep.fit_transform(X_tr)] + [prep.transform(df) for df in (X_va, X_te)]
    return [torch.tensor(a, dtype=torch.float32) for a in arrays]


Xt_tr, Xt_va, Xt_te = prepare_fold(*FOLDS[0])
print("input features after preprocessing:", Xt_tr.shape[1])
print("first fold tensors:", tuple(Xt_tr.shape), tuple(Xt_va.shape), tuple(Xt_te.shape))

## 모델 아키텍처

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_inputs: int, hidden=(256, 128), dropout: float = 0.1):
        super().__init__()
        layers, prev = [], n_inputs
        for size in hidden:
            layers += [nn.Linear(prev, size), nn.ReLU(), nn.Dropout(dropout)]
            prev = size
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)  # logits


print(MLP(Xt_tr.shape[1]))
print("trainable parameters:", sum(p.numel() for p in MLP(Xt_tr.shape[1]).parameters()))

## 학습 및 검증 루프

In [ ]:
BATCH_SIZE = 2048
MAX_EPOCHS = 40
PATIENCE = 4
LR = 2e-3


@torch.no_grad()
def predict(model, X_tensor):
    model.eval()
    out = [torch.sigmoid(model(X_tensor[i:i + 65_536].to(DEVICE))).cpu() for i in range(0, len(X_tensor), 65_536)]
    return torch.cat(out).numpy()


def train_fold(Xt_tr, y_tr, Xt_va, y_va, Xt_te):
    torch.manual_seed(SEED)
    model = MLP(Xt_tr.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss()
    yt_tr = torch.tensor(y_tr, dtype=torch.float32)

    best = {"auc": -1.0, "epoch": 0, "va_pred": None, "te_pred": None}
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        order = torch.randperm(len(Xt_tr))
        total_loss = 0.0
        for start in range(0, len(order), BATCH_SIZE):
            idx = order[start:start + BATCH_SIZE]
            xb, yb = Xt_tr[idx].to(DEVICE), yt_tr[idx].to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)

        va_pred = predict(model, Xt_va)
        va_auc = roc_auc_score(y_va, va_pred)
        history.append({"epoch": epoch, "train_loss": total_loss / len(order), "valid_auc": va_auc})
        if va_auc > best["auc"]:
            best = {"auc": va_auc, "epoch": epoch, "va_pred": va_pred, "te_pred": predict(model, Xt_te)}
        elif epoch - best["epoch"] >= PATIENCE:
            break
    return best, pd.DataFrame(history)

## Stratified 5-Fold 교차 검증 및 추론

In [ ]:
t0 = time.time()
oof, test_pred, fold_aucs, histories = np.full(len(X), np.nan), np.zeros(len(X_test)), [], []
for k, (tr_idx, va_idx) in enumerate(FOLDS):
    Xt_tr, Xt_va, Xt_te = prepare_fold(tr_idx, va_idx)
    best, history = train_fold(Xt_tr, y[tr_idx], Xt_va, y[va_idx], Xt_te)
    oof[va_idx] = best["va_pred"]
    test_pred += best["te_pred"] / N_FOLDS
    fold_aucs.append(best["auc"])
    histories.append(history)
    print(f"fold {k + 1}: AUC {best['auc']:.5f} at epoch {best['epoch']} (stopped at {len(history)}) | {time.time() - t0:.0f}s")

mlp_auc = roc_auc_score(y, oof)
print(f"\nMLP OOF AUC = {mlp_auc:.5f}")
save_predictions("mlp", oof, test_pred)

In [ ]:
from matplotlib.ticker import MaxNLocator

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for k in list(range(1, len(histories))) + [0]:  # draw fold 1 last so it stays on top
    h = histories[k]
    style = {"color": COLOR_MAIN if k == 0 else COLOR_MUTED, "linewidth": 2 if k == 0 else 1,
             "label": "fold 1" if k == 0 else ("other folds" if k == 1 else None)}
    axes[0].plot(h["epoch"], h["train_loss"], **style)
    axes[1].plot(h["epoch"], h["valid_auc"], **style)
    axes[1].scatter(h.loc[h["valid_auc"].idxmax(), "epoch"], h["valid_auc"].max(), s=25, color=style["color"], zorder=3)
axes[0].set_title("Training loss (lower is better)")
axes[1].set_title("Validation AUC (higher is better, dot = best epoch)")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()